In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA01 - Travel, Lodging, and Entertainment
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Coupa Data: 7 filtered files (ritm16070584_...)
   4. Finance Data: 6 files (f25_finance_data_...)
   
   BUSINESS LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + CANADA flag) will appear.
   2. SOURCE MERGE: Combines all 7 Coupa files and 6 Finance files into a single 
      unified transaction pipeline.
   3. STRING PARSING: Extracts the Cost Center and Category Code from Coupa's 
      hyphenated 'Account' string using split().
   4. TARGET CATEGORY CODES: Filters strictly for the ~30 approved target codes 
      provided in the master list.
   5. THE "793" EXCEPTION RULE: If a Category Code is 793, it is strictly locked to 
      Assessable Unit '101016'. If it maps to any other AU, it is intentionally dropped.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    -- 2. Pull the mapped Cost Centers from our finalized bootstrap view
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 3a. Stack all 7 Coupa target files together
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3b. Extract the CC (1st block) and Category Code (3rd block) from Coupa's hyphenated string
    SELECT 
        Account AS Raw_String,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    -- 4a. Stack all 6 Finance target files together
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- 4b. Standardize Finance columns to match Coupa format
    SELECT 
        CostCenter AS Raw_String,
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- 5. Merge the parsed Coupa and Finance streams into one master transaction list
    SELECT Raw_String, Cleaned_CC, CatCode_Int FROM Coupa_Parsed
    UNION ALL
    SELECT Raw_String, Cleaned_CC, CatCode_Int FROM Finance_Parsed
),

Mapped_Transactions AS (
    -- 6. Bridge all transactions to AUs
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        t.CatCode_Int,
        m.AU_ID
    FROM All_Transactions t
    INNER JOIN CC_Mapping m ON t.Cleaned_CC = m.Mapped_CC
),

Flagged_Engagements AS (
    -- 7. Apply the ~30 Category Code filters and the strict 793 exception rule
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        t.AU_ID
    FROM Mapped_Transactions t
    WHERE 
        -- RULE 1: Must be in the approved master category list
        t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
        
        -- RULE 2: If the code is 793, it MUST be mapped to AU 101016 to be counted
        AND NOT (t.CatCode_Int = 793 AND t.AU_ID != '101016')
),

Engagements_By_AU AS (
    -- 8. Count the total valid flagged transactions per AU
    SELECT 
        AU_ID,
        COUNT(Raw_String) AS Match_Count
    FROM Flagged_Engagements
    GROUP BY AU_ID
)

-- 9. Final Output: Strictly structured per Master Anchor using count of valid findings
SELECT 
    mast.BusinessID, 
    mast.AU_Name, 
    mast.Business_Segment,
    'EBA01' AS QuestionID,               
    COALESCE(e.Match_Count, 0) AS Response,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Engagements_By_AU e 
    ON mast.BusinessID = e.AU_ID
ORDER BY mast.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA01 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final count
   response was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    SELECT 
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        'Coupa' AS Source_System
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int,
        'Finance' AS Source_System
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    SELECT Cleaned_CC, CatCode_Int, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Cleaned_CC, CatCode_Int, Source_System FROM Finance_Parsed
),

Mapped_Transactions AS (
    SELECT 
        m.AU_ID,
        t.Cleaned_CC,
        t.CatCode_Int,
        t.Source_System
    FROM All_Transactions t
    INNER JOIN CC_Mapping m
        ON t.Cleaned_CC = m.Mapped_CC
),

Target_CatCode_Transactions AS (
    SELECT *
    FROM Mapped_Transactions
    WHERE CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
),

Counted_Transactions AS (
    SELECT *
    FROM Target_CatCode_Transactions
    WHERE NOT (CatCode_Int = 793 AND AU_ID != '101016')
),

Mapped_CC_List AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        COUNT(*) AS Mapped_Row_Count
    FROM Mapped_Transactions
    GROUP BY AU_ID
),

Target_CatCode_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Target_CatCode_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CAST(CatCode_Int AS STRING)))) AS Target_CatCode_List,
        COUNT(*) AS Target_CatCode_Row_Count
    FROM Target_CatCode_Transactions
    GROUP BY AU_ID
),

Counted_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Counted_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CAST(CatCode_Int AS STRING)))) AS Counted_CatCode_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Source_System))) AS Source_System_List,
        COUNT(*) AS Counted_Row_Count
    FROM Counted_Transactions
    GROUP BY AU_ID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA01' AS QuestionID,
    COALESCE(c.Counted_Row_Count, 0) AS Response,
    COALESCE(m.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(t.Target_CatCode_CC_List, 'n/a') AS Target_CatCode_CC_List,
    COALESCE(c.Counted_CC_List, 'n/a') AS Counted_CC_List,
    COALESCE(c.Counted_CatCode_List, 'n/a') AS Counted_CatCode_List,
    COALESCE(c.Source_System_List, 'n/a') AS Source_System_List,
    COALESCE(m.Mapped_Row_Count, 0) AS Mapped_Row_Count,
    COALESCE(t.Target_CatCode_Row_Count, 0) AS Target_CatCode_Row_Count,
    COALESCE(c.Counted_Row_Count, 0) AS Counted_Row_Count,
    COALESCE(c.Counted_Row_Count, 0) AS Final_Count,
    CONCAT('Response = ', CAST(COALESCE(c.Counted_Row_Count, 0) AS STRING), ' counted rows after target category filter and 793 exception rule. Target-category rows=', CAST(COALESCE(t.Target_CatCode_Row_Count, 0) AS STRING), ', mapped rows=', CAST(COALESCE(m.Mapped_Row_Count, 0) AS STRING), '.') AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Mapped_CC_List m
    ON mast.BusinessID = m.AU_ID
LEFT JOIN Target_CatCode_By_AU t
    ON mast.BusinessID = t.AU_ID
LEFT JOIN Counted_By_AU c
    ON mast.BusinessID = c.AU_ID
ORDER BY mast.BusinessID;
